# Retraining Cellpose on Custom Data

<div class="custom-button-row">
    <a 
        class="custom-button custom-download-button" href="../../../notebooks/05_segmentation/deep_learning/cellpose_retraining_colab.ipynb" download>
        <i class="fas fa-download"></i> Download this Notebook
    </a>
    <a
    class="custom-button custom-download-button" href="https://colab.research.google.com/github/bobiac/bobiac-book/blob/gh-pages/colab_notebooks/05_segmentation/deep_learning/cellpose_retraining_colab.ipynb" target="_blank">
        <img class="button-icon" src="../../../_static/logo/icon-google-colab.svg" alt="Open in Colab">
        Open in Colab
    </a>
</div>

In [1]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "matplotlib",
#     "cellpose",
#     "tqdm"
# ]
# ///

## Overview

In this section, we’ll walk through how to **retrain Cellpose on your own data**. This is useful when the default models don’t perform well on your specific cell type, staining method, or imaging modality.

Retraining allows Cellpose to learn directly from your examples—leading to better segmentation accuracy and more relevant masks for your experiments.

We’ll cover:
- Preparing your training data (images + label masks)
- Mounting your Google Drive to access files
- Setting training parameters
- Running the training process
- Evaluating the new model on test images


‼ MENTION NAPARI

> 💡 You’ll need pairs of raw microscopy images and their corresponding label masks. If you haven’t labeled your images yet, we recommend using the [Cellpose GUI](https://cellpose.readthedocs.io/en/latest/gui.html#training-your-own-cellpose-model) to draw or edit masks manually before starting.

The dataset we’ll use here can be downloaded below. It includes both training and test images:
<a href="../../../_static/data/05_segmentation_cellpose_training.zip" download>
<i class="fas fa-download"></i> Cellpose Training Dataset</a>

<p class="alert alert-warning">
    <strong>⚠️ Note:</strong> This notebook is designed to run in <a href="https://colab.research.google.com/github/bobiac/bobiac-book/blob/gh-pages/colab_notebooks/05_segmentation/deep_learning/cellpose_retraining_notebook.ipynb" target="_blank"> Google Colab</a>. If you want to run it locally, you may need to adjust some paths and install the required packages.
</p>

## Import Libraries

In [2]:
from pathlib import Path

from cellpose import core, io, metrics, models, train

## Setup

In [3]:
io.logger_setup()  # to get printing of progress

use_gpu = core.use_gpu()
print("GPU available:", use_gpu)

2026-06-04 11:21:24,272 [INFO] WRITING LOG OUTPUT TO /Users/fdrgsp/.cellpose/run.log
2026-06-04 11:21:24,273 [INFO] 
cellpose version: 	4.1.1 
platform:       	darwin 
python version: 	3.13.3 
torch version:  	2.12.0
2026-06-04 11:21:24,296 [INFO] ** TORCH MPS version installed and working. **
GPU available: True


## Data Handling

Cellpose expects images and their corresponding masks to live **in the same folder**, paired by a shared base name. Each image and mask share a numeric prefix — images have an `_img` suffix, masks have a `_masks` suffix.

Both the `train` and `test` folders follow the same layout:

```
cellpose_data/
├── train/
│   ├── 000_img.tiff
│   ├── 000_masks.tif
│   ├── 001_img.tiff
│   ├── 001_masks.tif
│   └── ...
└── test/
    ├── 008_img.tiff
    ├── 008_masks.tif
    ├── 009_img.tiff
    ├── 009_masks.tif
    └── ...
```

You'll also need to **split your data** into a `train` and a `test` folder. The model learns from the training set and is evaluated on the test set—images it has never seen during training.

> 💡 If you haven't labeled your images yet, the [Cellpose GUI](https://cellpose.readthedocs.io/en/latest/gui.html#training-your-own-cellpose-model) lets you draw or correct masks manually and saves them with the `_masks` suffix automatically.

The first step is to define the `train` and `test` directories, then load the data with [`io.load_train_test_data`](https://cellpose.readthedocs.io/en/latest/api.html#cellpose.io.load_train_test_data), passing both the image and mask suffixes so Cellpose can pair them correctly.

In [14]:
# ROOT_FOLDER_PATH = Path("data/05_segmentation_cellpose/retraining")
ROOT_FOLDER_PATH = Path("/Users/fdrgsp/Desktop/cellpose_retraining")

train_dir = ROOT_FOLDER_PATH / "train"
test_dir = ROOT_FOLDER_PATH / "test"

# suffix that identifies image files (e.g. "000_img.tiff")
image_filter = "_img"
# suffix that identifies mask files (e.g. "000_masks.tif")
mask_filter = "_masks"

# Load training and test data
train_data, train_labels, _, test_data, test_labels, _ = io.load_train_test_data(
    str(train_dir), str(test_dir), image_filter=image_filter, mask_filter=mask_filter
)

2026-06-04 15:12:25,651 [INFO] not all flows are present, running flow generation for all images
2026-06-04 15:12:25,664 [INFO] 6 / 6 images in /Users/fdrgsp/Desktop/cellpose_retraining/train folder have labels
2026-06-04 15:12:25,665 [INFO] not all flows are present, running flow generation for all images
2026-06-04 15:12:25,669 [INFO] 2 / 2 images in /Users/fdrgsp/Desktop/cellpose_retraining/test folder have labels


During training, Cellpose will:
- Load batches of training images
- Compare its predictions to the ground-truth masks
- Adjust itself (via backpropagation) to reduce errors over time

### Init the Model

Before we can train a new model, we need to initialize Cellpose with the correct settings.

Here, we’ll:
- Specify the **model type** (e.g., "cpsam" (default), "cyto" or "nuclei") to use as a base model
- Set the **channels** depending on how your images are structured (e.g., single-channel grayscale, or dual-channel with nuclei and cytoplasm)
- Choose where to **save the model weights** during training

> 💡 Even when training a new model, Cellpose builds on a pre-trained backbone (unless you explicitly start from scratch). This helps it learn faster and perform better—especially on small datasets.


In [ ]:
# Initialize the Cellpose model
model = models.CellposeModel(model_type="cpsam", gpu=use_gpu)

2026-06-04 15:13:36,400 [WARNING] model_type argument is not used in v4.0.1+. Ignoring this argument...
2026-06-04 15:13:36,406 [INFO] ** TORCH MPS version installed and working. **
2026-06-04 15:13:36,407 [INFO] >>>> using GPU (MPS)
2026-06-04 15:13:37,170 [INFO] >>>> loading model /Users/fdrgsp/.cellpose/models/cpsam


In [ ]:
# run model on test images
masks = model.eval(test_data, batch_size=32)[0]

# check performance using ground truth labels
# average_precision returns AP at IoU thresholds [0.5, 0.75, 0.9] by default
ap = metrics.average_precision(test_labels, masks)[0]
print(f"average precision at iou threshold 0.5  = {ap[:, 0].mean():.3f}")
print(f"average precision at iou threshold 0.75 = {ap[:, 1].mean():.3f}")
print(f"average precision at iou threshold 0.9  = {ap[:, 2].mean():.3f}")

average precision at iou threshold 0.5  = 0.893
average precision at iou threshold 0.75 = 0.671
average precision at iou threshold 0.9  = 0.179


## Train New Model

Now we’re ready to train! In this step, we’ll tell Cellpose to:
- Use the training images and masks
- Save the trained model to your specified directory
- Run for a defined number of **epochs** (iterations over the full dataset)

You can also set other options like:
- Learning rate
- Batch size
- Whether to use GPU

> 💡 Training time will vary depending on your dataset size and hardware. On Google Colab with a GPU, small datasets may train in just a few minutes.

After training, the model weights will be saved and ready to use for predictions. We’ll evaluate performance on the test data in the next step.


In [18]:
model_name = "new_model"

# Training params
n_epochs = 10
learning_rate = 1e-5
weight_decay = 0.1
batch_size = 1

# (not passing test data into function to speed up training)

new_model_path, train_losses, test_losses = train.train_seg(
    model.net,
    train_data=train_data,
    train_labels=train_labels,
    batch_size=batch_size,
    n_epochs=n_epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    nimg_per_epoch=max(2, len(train_data)),  # can change this
    model_name=model_name,
)

2026-06-04 15:18:34,166 [INFO] >>> converting bfloat16 network to float32 for training
2026-06-04 15:18:34,392 [INFO] computing flows for labels


100%|██████████| 6/6 [00:00<00:00,  9.00it/s]

2026-06-04 15:18:35,062 [INFO] >>> computing diameters



100%|██████████| 6/6 [00:00<00:00, 1738.09it/s]

2026-06-04 15:18:35,067 [INFO] >>> normalizing {'lowhigh': None, 'percentile': None, 'normalize': True, 'norm3D': True, 'sharpen_radius': 0, 'smooth_radius': 0, 'tile_norm_blocksize': 0, 'tile_norm_smooth3D': 1, 'invert': False}


2026-06-04 15:18:35,084 [INFO] >>> n_epochs=10, n_train=6, n_test=None
2026-06-04 15:18:35,084 [INFO] >>> AdamW, learning_rate=0.00001, weight_decay=0.10000
2026-06-04 15:18:35,086 [INFO] >>> saving model to /Users/fdrgsp/Documents/git/bobiac-book/content/05_segmentation/deep_learning/models/new_model


/Users/fdrgsp/Documents/git/bobiac-book/.venv/lib/python3.13/site-packages/cellpose/train.py:463: UserWarning: In MPS autocast, but the target dtype is not supported. Disabling autocast.
MPS Autocast only supports dtypes of torch.bfloat16, torch.float16 currently.
  with torch.autocast(device_type=device.type, dtype=net.dtype):


2026-06-04 15:18:44,050 [INFO] 0, train_loss=0.2591, test_loss=0.0000, LR=0.000000, time 8.96s
2026-06-04 15:19:10,264 [INFO] 5, train_loss=0.2242, test_loss=0.0000, LR=0.000006, time 35.18s
2026-06-04 15:19:31,198 [INFO] saving network parameters to /Users/fdrgsp/Documents/git/bobiac-book/content/05_segmentation/deep_learning/models/new_model
2026-06-04 15:19:33,615 [INFO] >>> converting network back to torch.bfloat16 after training


In [21]:
test_losses

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

## Evaluate on test data

In [22]:
model = models.CellposeModel(pretrained_model=new_model_path, gpu=use_gpu)

# run model on test images
masks = model.eval(test_data, batch_size=32)[0]

# check performance using ground truth labels
# average_precision returns AP at IoU thresholds [0.5, 0.75, 0.9] by default
ap = metrics.average_precision(test_labels, masks)[0]
print(f"average precision at iou threshold 0.5  = {ap[:, 0].mean():.3f}")
print(f"average precision at iou threshold 0.75 = {ap[:, 1].mean():.3f}")
print(f"average precision at iou threshold 0.9  = {ap[:, 2].mean():.3f}")

2026-06-04 15:21:39,598 [INFO] ** TORCH MPS version installed and working. **
2026-06-04 15:21:39,598 [INFO] >>>> using GPU (MPS)
2026-06-04 15:21:40,292 [INFO] >>>> loading model /Users/fdrgsp/Documents/git/bobiac-book/content/05_segmentation/deep_learning/models/new_model
2026-06-04 15:21:44,578 [INFO] 100%|##########| 2/2 [00:03<00:00,  1.97s/it]
average precision at iou threshold 0.5  = 0.876
average precision at iou threshold 0.75 = 0.659
average precision at iou threshold 0.9  = 0.179
